In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv('/content/anime.csv')
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [3]:
# Convert to numeric
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['members'] = pd.to_numeric(df['members'], errors='coerce')

In [4]:
# Handle missing values
df['genre'] = df['genre'].fillna('')
df = df.dropna(subset=['episodes', 'rating', 'members'])

In [5]:
# Reset index
df = df.reset_index(drop=True)

In [6]:
print(df.shape)

(11876, 7)


In [7]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [8]:
# TF-IDF on train
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix_train = tfidf.fit_transform(train_df['genre'])

In [9]:
# Numeric scaling
scaler = MinMaxScaler()
numeric_train = scaler.fit_transform(train_df[['rating', 'episodes', 'members']])

In [10]:
numeric_train_sparse = csr_matrix(numeric_train)

In [11]:
# Combine
feature_matrix_train = hstack([genre_matrix_train, numeric_train_sparse])

In [12]:
cosine_sim_train = cosine_similarity(feature_matrix_train, feature_matrix_train)

In [13]:
def recommend_with_threshold(title, threshold=0.5):

    if title not in train_df['name'].values:
        print("Anime not found")
        return []

    idx = train_df[train_df['name'] == title].index[0]

    sim_scores = list(enumerate(cosine_sim_train[idx]))

    sim_scores = [i for i in sim_scores if i[1] >= threshold]

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:]

    indices = [i[0] for i in sim_scores]

    return train_df['name'].iloc[indices]

In [14]:
recommend_with_threshold("Naruto", 0.5)[:10]

,name
5949,Dragon Ball Z
3278,Dragon Ball
2829,Naruto: Shippuuden Movie 4 - The Lost Tower
5461,Boruto: Naruto the Movie
1113,Naruto x UT
7280,Naruto Soyokazeden Movie: Naruto to Mashin to ...
7817,Boruto: Naruto the Movie - Naruto ga Hokage ni...
4299,Naruto Shippuuden: Sunny Side Battle
6845,Naruto: Shippuuden Movie 6 - Road to Ninja
8704,The Last: Naruto the Movie


In [15]:
print("Threshold 0.3:\n", recommend_with_threshold("Naruto", 0.3)[:10])
print("\nThreshold 0.5:\n", recommend_with_threshold("Naruto", 0.5)[:10])
print("\nThreshold 0.7:\n", recommend_with_threshold("Naruto", 0.7)[:10])

Threshold 0.3:
 5949                                        Dragon Ball Z
3278                                          Dragon Ball
2829          Naruto: Shippuuden Movie 4 - The Lost Tower
5461                             Boruto: Naruto the Movie
1113                                          Naruto x UT
7280    Naruto Soyokazeden Movie: Naruto to Mashin to ...
7817    Boruto: Naruto the Movie - Naruto ga Hokage ni...
4299                 Naruto Shippuuden: Sunny Side Battle
6845           Naruto: Shippuuden Movie 6 - Road to Ninja
8704                           The Last: Naruto the Movie
Name: name, dtype: object

Threshold 0.5:
 5949                                        Dragon Ball Z
3278                                          Dragon Ball
2829          Naruto: Shippuuden Movie 4 - The Lost Tower
5461                             Boruto: Naruto the Movie
1113                                          Naruto x UT
7280    Naruto Soyokazeden Movie: Naruto to Mashin to ...
7817    Borut

In [16]:
def evaluate_system(threshold=0.5):
    precisions = []

    for i in range(len(test_df)):
        title = test_df['name'].iloc[i]
        recommended = recommend_with_threshold(title, threshold)

        if len(recommended) == 0:
            continue

        actual_genre = set(test_df['genre'].iloc[i].split(', '))

        match = 0
        for rec in recommended:
            rec_genre = set(train_df[train_df['name'] == rec]['genre'].values[0].split(', '))
            if actual_genre.intersection(rec_genre):
                match += 1

        precision = match / len(recommended)
        precisions.append(precision)

    if len(precisions) == 0:
        print(f"Threshold: {threshold} → No recommendations")
    else:
        print(f"Threshold: {threshold}")
        print("Precision:", np.mean(precisions))

In [17]:
print("Naruto" in train_df['name'].values)

True


In [18]:
train_df['name'].iloc[0]

'Choujin Locke: Shinsekai Sentai'

In [19]:
anime_name = train_df['name'].iloc[0]

print("Using:", anime_name)

print("\nThreshold 0.3:\n", recommend_with_threshold(anime_name, 0.3)[:5])
print("\nThreshold 0.5:\n", recommend_with_threshold(anime_name, 0.5)[:5])
print("\nThreshold 0.7:\n", recommend_with_threshold(anime_name, 0.7)[:5])

Using: Choujin Locke: Shinsekai Sentai

Threshold 0.3:
 1689      Choujin Locke: Lord Leon
3582    Choujin Locke: Mirror Ring
819                        Ai City
5764                 Choujin Locke
4956                     Ougon Bat
Name: name, dtype: object

Threshold 0.5:
 1689      Choujin Locke: Lord Leon
3582    Choujin Locke: Mirror Ring
819                        Ai City
5764                 Choujin Locke
4956                     Ougon Bat
Name: name, dtype: object

Threshold 0.7:
 1689      Choujin Locke: Lord Leon
3582    Choujin Locke: Mirror Ring
819                        Ai City
5764                 Choujin Locke
4956                     Ougon Bat
Name: name, dtype: object


## Observation

It was observed that for some anime, the recommendation results remained the same across different threshold values (0.3, 0.5, and 0.7).

This indicates that the similarity scores between these anime are very high, exceeding even the highest threshold value.

### Conclusion:

For highly similar items, changing the threshold does not affect the recommendation list.

This shows that the recommendation system is effectively capturing strong similarities between anime.


# Interview Questions:

1. Can you explain the difference between user-based and item-based collaborative filtering?

User-Based:

Finds similar users

Recommends what similar users like

Item-Based:

Finds similar items

Recommends similar items

User-based is slower and less scalable, while item-based is faster and more efficient.

2. What is collaborative filtering, and how does it work?

Collaborative filtering is a recommendation technique that suggests items based on user behavior.

How it works:

Create user-item matrix,
Compute similarity

Recommend based on similar users/items

Types:

User-Based,
Item-Based

Problems:

Cold start problem,
Data sparsity